# 📊 MODULE 5: INTEGRATION & PRACTICE
## Time Series Analytics - Educational Notebook for Google Colab

---

### Topics Covered:
- **5.1** End-to-End Time Series Analysis Pipeline
- **5.2** Reusable Model Comparison Function
- **5.3** Interactive Visualization with Plotly

---

### Learning Objectives:
This module integrates all concepts from previous modules into:
1. A **complete analysis pipeline** you can apply to any time series
2. **Reusable functions** for quick model comparison
3. **Interactive visualizations** for better data exploration

---

**Instructions:** Run each cell in order. This module builds a complete toolkit for time series analysis.

## 🔧 Setup: Install and Import Required Packages

Run this cell first to install dependencies and download data.

In [ ]:
# Install required packages (uncomment if running on Colab for the first time)
# !pip install yfinance plotly --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljung_box
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy import stats

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All packages imported successfully!")

In [ ]:
# Download data
print("Downloading data...")
btc = yf.download('BTC-USD', start='2020-01-01', progress=False)['Close'].dropna()
gold = yf.download('GC=F', start='2020-01-01', progress=False)['Close'].dropna()
mrf = yf.download('MRF.NS', start='2020-01-01', progress=False)['Close'].dropna()

print(f"✓ Bitcoin: {len(btc)} observations")
print(f"✓ Gold: {len(gold)} observations")
print(f"✓ MRF: {len(mrf)} observations")
print("\n✅ Setup complete! Ready to run Module 5 topics.")

---

# 📌 TOPIC 5.1: End-to-End Time Series Analysis Pipeline

### Complete Workflow:
A systematic approach to time series analysis following these steps:

| Step | Task | Purpose |
|------|------|--------|
| 1 | Data Loading | Import and prepare data |
| 2 | EDA | Visualize and understand patterns |
| 3 | Decomposition | Separate trend, seasonal, residual |
| 4 | Stationarity Test | Check if differencing needed |
| 5 | Model Identification | Use ACF/PACF to determine orders |
| 6 | Model Fitting | Train multiple candidate models |
| 7 | Forecasting | Generate predictions |
| 8 | Diagnostics | Validate residuals |

Let's apply this pipeline to Bitcoin price data!

### Step 1: Data Loading and Preparation

In [ ]:
# STEP 1: Load and prepare data
data = btc.copy()

print("STEP 1: Data Loading")
print("="*60)
print(f"  Dataset: Bitcoin (BTC-USD)")
print(f"  Observations: {len(data)}")
print(f"  Date Range: {data.index[0].date()} to {data.index[-1].date()}")
print(f"  Missing Values: {data.isna().sum()}")
print(f"\n  Basic Statistics:")
print(f"    Min: ${data.min():,.2f}")
print(f"    Max: ${data.max():,.2f}")
print(f"    Mean: ${data.mean():,.2f}")
print(f"    Std Dev: ${data.std():,.2f}")

### Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# STEP 2: Exploratory Data Analysis
print("\nSTEP 2: Exploratory Data Analysis")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time series plot
axes[0, 0].plot(data, color='blue', linewidth=1)
axes[0, 0].set_title('Bitcoin Price Over Time', fontweight='bold')
axes[0, 0].set_ylabel('Price (USD)')
axes[0, 0].grid(True, alpha=0.3)

# Distribution
axes[0, 1].hist(data, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 1].set_title('Price Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Price (USD)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# ACF
plot_acf(data, lags=40, ax=axes[1, 0])
axes[1, 0].set_title('Autocorrelation Function (ACF)', fontweight='bold')

# PACF
plot_pacf(data, lags=40, ax=axes[1, 1])
axes[1, 1].set_title('Partial Autocorrelation Function (PACF)', fontweight='bold')

plt.tight_layout()
plt.show()

print("  ✓ Data visualized")
print("  ✓ Strong autocorrelation observed → likely non-stationary")

### Step 3: Time Series Decomposition

In [ ]:
# STEP 3: Decomposition
print("\nSTEP 3: Time Series Decomposition")
print("="*60)

decomposition = seasonal_decompose(data, model='multiplicative', period=365)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))

decomposition.observed.plot(ax=axes[0], color='blue', linewidth=1)
axes[0].set_ylabel('Observed')
axes[0].set_title('Bitcoin Price Decomposition (Multiplicative)', fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3)

decomposition.trend.plot(ax=axes[1], color='red', linewidth=2)
axes[1].set_ylabel('Trend')
axes[1].grid(True, alpha=0.3)

decomposition.seasonal.plot(ax=axes[2], color='green', linewidth=1)
axes[2].set_ylabel('Seasonal')
axes[2].grid(True, alpha=0.3)

decomposition.resid.plot(ax=axes[3], color='purple', linewidth=1)
axes[3].set_ylabel('Residual')
axes[3].set_xlabel('Date')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("  ✓ Decomposition complete")
print("  ✓ Clear upward trend identified")
print("  ✓ Seasonal patterns visible")

### Step 4: Stationarity Testing

In [ ]:
# STEP 4: Check Stationarity
print("\nSTEP 4: Stationarity Testing")
print("="*60)

adf_original = adfuller(data)
print(f"\n  ADF Test on Original Data:")
print(f"    Test Statistic: {adf_original[0]:.4f}")
print(f"    p-value: {adf_original[1]:.4f}")
print(f"    Critical Values:")
for key, value in adf_original[4].items():
    print(f"      {key}: {value:.4f}")

if adf_original[1] > 0.05:
    print(f"\n    → Non-stationary (p > 0.05)")
    print(f"\n  Applying first differencing...")
    
    # Make stationary
    data_diff = data.diff().dropna()
    
    adf_diff = adfuller(data_diff)
    print(f"\n  ADF Test on Differenced Data:")
    print(f"    Test Statistic: {adf_diff[0]:.4f}")
    print(f"    p-value: {adf_diff[1]:.4f}")
    
    if adf_diff[1] < 0.05:
        print(f"    ✓ Stationary after differencing (p < 0.05)")
        stationary_data = data_diff
        d_order = 1
    else:
        print(f"    Applying second differencing...")
        data_diff2 = data_diff.diff().dropna()
        adf_diff2 = adfuller(data_diff2)
        print(f"    p-value after 2nd diff: {adf_diff2[1]:.4f}")
        stationary_data = data_diff2
        d_order = 2
else:
    print(f"    ✓ Already stationary")
    stationary_data = data
    d_order = 0

print(f"\n  → Differencing order selected: d = {d_order}")

### Step 5: Model Identification Using ACF/PACF

In [ ]:
# STEP 5: Identify Model Using ACF/PACF
print(f"\nSTEP 5: Model Identification (d={d_order})")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(stationary_data, lags=40, ax=axes[0])
axes[0].set_title('ACF of Stationary Data', fontweight='bold')

plot_pacf(stationary_data, lags=40, ax=axes[1])
axes[1].set_title('PACF of Stationary Data', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n  Analyzing ACF/PACF patterns...")
print("  Based on patterns, testing ARIMA models with p,q ∈ {0,1,2}")
print("\n  Model Identification Rules:")
print("    - ACF cuts off, PACF decays → MA(q) model")
print("    - ACF decays, PACF cuts off → AR(p) model")
print("    - Both decay → ARMA(p,q) model")

### Step 6: Model Fitting and Selection

In [ ]:
# STEP 6: Fit and Validate Models
print("\nSTEP 6: Model Fitting and Selection")
print("="*60)

# Split data
train_size = int(len(data) * 0.8)
train = data[:train_size]
test = data[train_size:]

print(f"\n  Training set: {len(train)} observations")
print(f"  Test set: {len(test)} observations")

# Test multiple models
candidate_models = [(1,d_order,1), (2,d_order,1), (1,d_order,2), (2,d_order,2)]
model_results = []

print(f"\n  Testing candidate models...")
for order in candidate_models:
    try:
        model = ARIMA(train, order=order)
        fitted = model.fit()
        
        # Forecast
        forecast = fitted.forecast(steps=len(test))
        rmse = np.sqrt(np.mean((forecast.values - test.values)**2))
        
        model_results.append({
            'Order': f"ARIMA{order}",
            'AIC': fitted.aic,
            'BIC': fitted.bic,
            'Test RMSE': rmse
        })
        print(f"    ✓ ARIMA{order} fitted")
    except Exception as e:
        print(f"    ✗ ARIMA{order} failed: {e}")

results_df = pd.DataFrame(model_results).sort_values('AIC')
print("\n  Model Comparison:")
print(results_df.to_string(index=False))

best_model_info = results_df.iloc[0]
best_order = eval(best_model_info['Order'].replace('ARIMA', ''))

print(f"\n  ✓ Best model selected: {best_model_info['Order']}")

### Step 7: Final Model and Forecast

In [ ]:
# STEP 7: Fit Best Model and Forecast
print("\nSTEP 7: Final Model and Forecast")
print("="*60)

final_model = ARIMA(train, order=best_order)
final_fitted = final_model.fit()

print(f"\n  Model: ARIMA{best_order}")
print(f"  AIC: {final_fitted.aic:.2f}")
print(f"  BIC: {final_fitted.bic:.2f}")

# Forecast
forecast_final = final_fitted.forecast(steps=len(test))
forecast_ci = final_fitted.get_forecast(steps=len(test)).conf_int()

In [ ]:
# Plot forecast
plt.figure(figsize=(14, 6))
plt.plot(train.index[-200:], train[-200:], label='Training Data', color='blue', linewidth=2)
plt.plot(test.index, test, label='Actual Test Data', color='green', linewidth=2)
plt.plot(test.index, forecast_final, label='Forecast', color='red', linewidth=2, linestyle='--')
plt.fill_between(test.index, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1],
                 color='red', alpha=0.2, label='95% Confidence Interval')
plt.title(f'Final Model: ARIMA{best_order} Forecast', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Bitcoin Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate forecast accuracy
rmse = np.sqrt(np.mean((forecast_final.values - test.values)**2))
mae = np.mean(np.abs(forecast_final.values - test.values))
mape = np.mean(np.abs((test.values - forecast_final.values) / test.values)) * 100

print(f"\n  Forecast Accuracy Metrics:")
print(f"    RMSE: ${rmse:,.2f}")
print(f"    MAE: ${mae:,.2f}")
print(f"    MAPE: {mape:.2f}%")

### Step 8: Residual Diagnostics

In [ ]:
# STEP 8: Residual Diagnostics
print("\nSTEP 8: Residual Diagnostics")
print("="*60)

residuals = final_fitted.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals plot
axes[0, 0].plot(residuals, color='blue', linewidth=0.8)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_title('Residuals Over Time', fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Residual')
axes[0, 0].grid(True, alpha=0.3)

# ACF of residuals
plot_acf(residuals, lags=30, ax=axes[0, 1])
axes[0, 1].set_title('ACF of Residuals', fontweight='bold')

# Histogram
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='steelblue', density=True)
# Overlay normal distribution
x = np.linspace(residuals.min(), residuals.max(), 100)
axes[1, 0].plot(x, stats.norm.pdf(x, residuals.mean(), residuals.std()), 'r-', lw=2, label='Normal')
axes[1, 0].set_title('Residual Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Residual')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Ljung-Box Test
lb_test = acorr_ljung_box(residuals, lags=[10], return_df=True)
print(f"\n  Ljung-Box Test (lag=10):")
print(f"    Test Statistic: {lb_test['lb_stat'].values[0]:.4f}")
print(f"    p-value: {lb_test['lb_pvalue'].values[0]:.4f}")

if lb_test['lb_pvalue'].values[0] > 0.05:
    print("    ✓ Residuals appear to be white noise (good!)")
else:
    print("    ⚠ Some autocorrelation remains in residuals")

### Pipeline Summary

In [ ]:
print("\n" + "="*60)
print("END-TO-END PIPELINE COMPLETE!")
print("="*60)
print("\nSummary:")
print(f"  1. ✓ Data loaded and explored ({len(data)} observations)")
print(f"  2. ✓ Decomposition revealed trend and seasonality")
print(f"  3. ✓ Made data stationary with {d_order} differencing")
print(f"  4. ✓ Identified model using ACF/PACF")
print(f"  5. ✓ Tested {len(candidate_models)} ARIMA specifications")
print(f"  6. ✓ Selected best model: ARIMA{best_order}")
print(f"  7. ✓ Generated forecast with confidence intervals")
print(f"  8. ✓ Validated model through residual diagnostics")

### 📝 Key Takeaway:
This 8-step pipeline provides a **systematic approach** to time series analysis:
1. Always start with **data exploration**
2. Check for **stationarity** before modeling
3. Use **ACF/PACF** to guide model selection
4. Compare **multiple models** using AIC/BIC and test RMSE
5. Always **validate** with residual diagnostics

---

# 📌 TOPIC 5.2: Reusable Model Comparison Function

### Key Concepts:
Create a **reusable function** that:
- Takes any time series data
- Fits multiple ARIMA models
- Returns comparison metrics
- Generates visualization

This saves time when analyzing multiple datasets!

In [ ]:
def compare_models(data, models_dict, train_ratio=0.8):
    """
    Compare multiple time series models on given data.
    
    Parameters:
    -----------
    data : pd.Series
        Time series data
    models_dict : dict
        Dictionary of model_name: ARIMA_order pairs
        Example: {'AR(2)': (2,0,0), 'ARIMA(1,1,1)': (1,1,1)}
    train_ratio : float
        Proportion of data for training (default 0.8)
    
    Returns:
    --------
    pd.DataFrame
        Comparison results with metrics for each model
    """
    # Split data
    split_point = int(len(data) * train_ratio)
    train = data[:split_point]
    test = data[split_point:]
    
    results = []
    forecasts = {}
    
    for name, order in models_dict.items():
        try:
            # Fit model
            model = ARIMA(train, order=order)
            fitted = model.fit()
            
            # Forecast
            forecast = fitted.forecast(steps=len(test))
            forecasts[name] = forecast
            
            # Calculate metrics
            rmse = np.sqrt(np.mean((forecast.values - test.values)**2))
            mae = np.mean(np.abs(forecast.values - test.values))
            mape = np.mean(np.abs((test.values - forecast.values) / test.values)) * 100
            
            results.append({
                'Model': name,
                'Order': str(order),
                'AIC': fitted.aic,
                'BIC': fitted.bic,
                'RMSE': rmse,
                'MAE': mae,
                'MAPE (%)': mape
            })
        except Exception as e:
            print(f"  Failed to fit {name}: {str(e)}")
    
    results_df = pd.DataFrame(results)
    
    # Plot comparison
    plt.figure(figsize=(14, 6))
    plt.plot(test.index, test, label='Actual', color='black', linewidth=2)
    
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    for i, (name, forecast) in enumerate(forecasts.items()):
        plt.plot(test.index, forecast, label=name, 
                color=colors[i % len(colors)], linewidth=1.5, alpha=0.7)
    
    plt.title('Model Comparison: Forecasts', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return results_df

print("✓ compare_models() function defined!")
print("\nFunction signature:")
print("  compare_models(data, models_dict, train_ratio=0.8)")
print("\nReturns: DataFrame with AIC, BIC, RMSE, MAE, MAPE for each model")

### Testing the Function on Multiple Datasets

In [ ]:
# Define models to compare
test_models = {
    'AR(2)': (2, 0, 0),
    'MA(2)': (0, 0, 2),
    'ARMA(1,1)': (1, 0, 1),
    'ARIMA(1,1,1)': (1, 1, 1),
    'ARIMA(2,1,2)': (2, 1, 2)
}

print("Models to compare:")
for name, order in test_models.items():
    print(f"  {name}: {order}")

In [ ]:
# Test on Bitcoin
print("\n" + "="*60)
print("Dataset 1: Bitcoin")
print("="*60)
btc_results = compare_models(btc, test_models)
print(btc_results.to_string(index=False))
print(f"\n✓ Best model by RMSE: {btc_results.loc[btc_results['RMSE'].idxmin(), 'Model']}")

In [ ]:
# Test on Gold
print("\n" + "="*60)
print("Dataset 2: Gold")
print("="*60)
gold_results = compare_models(gold, test_models)
print(gold_results.to_string(index=False))
print(f"\n✓ Best model by RMSE: {gold_results.loc[gold_results['RMSE'].idxmin(), 'Model']}")

In [ ]:
# Test on MRF
print("\n" + "="*60)
print("Dataset 3: MRF Stock")
print("="*60)
mrf_results = compare_models(mrf, test_models)
print(mrf_results.to_string(index=False))
print(f"\n✓ Best model by RMSE: {mrf_results.loc[mrf_results['RMSE'].idxmin(), 'Model']}")

### 📝 Key Takeaway:
- **Reusable functions** save time and ensure consistency
- The same function works on any time series dataset
- Different datasets may have different "best" models
- Always test multiple models before selecting the final one

---

# 📌 TOPIC 5.3: Interactive Visualization with Plotly

### Key Concepts:
**Interactive visualizations** allow you to:
- Hover to see exact values
- Zoom in/out to explore patterns
- Pan to navigate through time
- Download as static images

We'll use **Plotly** for creating interactive charts.

In [ ]:
# Install plotly if needed
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    print("✓ Plotly available")
except:
    print("Installing plotly...")
    !pip install plotly --quiet
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    print("✓ Plotly installed")

### Interactive Time Series Chart

In [ ]:
# Interactive Time Series Plot
print("Creating interactive Bitcoin price chart...")

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=btc.index,
    y=btc.values,
    mode='lines',
    name='Bitcoin Price',
    line=dict(color='blue', width=2),
    hovertemplate='<b>Date</b>: %{x}<br><b>Price</b>: $%{y:,.2f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Interactive Bitcoin Price Chart',
        font=dict(size=18, color='black')
    ),
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    hovermode='x unified',
    template='plotly_white',
    height=500,
    xaxis=dict(
        rangeslider=dict(visible=True),
        type='date'
    )
)

fig.show()
print("✓ Interactive chart created!")
print("  → Hover to see values")
print("  → Use range slider to zoom")
print("  → Double-click to reset")

### Interactive Decomposition Plot

In [ ]:
# Interactive Decomposition
print("\nCreating interactive decomposition plot...")

decomp = seasonal_decompose(btc, model='multiplicative', period=365)

fig_decomp = make_subplots(
    rows=4, cols=1,
    subplot_titles=('Observed', 'Trend', 'Seasonal', 'Residual'),
    vertical_spacing=0.08,
    shared_xaxes=True
)

fig_decomp.add_trace(
    go.Scatter(x=btc.index, y=decomp.observed, name='Observed', 
               line=dict(color='blue'), hovertemplate='%{y:,.2f}<extra></extra>'),
    row=1, col=1
)

fig_decomp.add_trace(
    go.Scatter(x=btc.index, y=decomp.trend, name='Trend', 
               line=dict(color='red', width=2), hovertemplate='%{y:,.2f}<extra></extra>'),
    row=2, col=1
)

fig_decomp.add_trace(
    go.Scatter(x=btc.index, y=decomp.seasonal, name='Seasonal', 
               line=dict(color='green'), hovertemplate='%{y:.4f}<extra></extra>'),
    row=3, col=1
)

fig_decomp.add_trace(
    go.Scatter(x=btc.index, y=decomp.resid, name='Residual', 
               line=dict(color='purple'), hovertemplate='%{y:.4f}<extra></extra>'),
    row=4, col=1
)

fig_decomp.update_layout(
    height=900,
    showlegend=False,
    title_text="Interactive Time Series Decomposition",
    hovermode='x unified',
    template='plotly_white'
)

fig_decomp.show()
print("✓ Interactive decomposition created!")

### Interactive ACF Plot

In [ ]:
# Interactive ACF Plot
print("\nCreating interactive ACF plot...")

acf_values = acf(btc, nlags=40)
lags = list(range(len(acf_values)))

fig_acf = go.Figure()

# ACF bars
fig_acf.add_trace(go.Bar(
    x=lags,
    y=acf_values,
    name='ACF',
    marker_color='steelblue',
    hovertemplate='<b>Lag</b>: %{x}<br><b>ACF</b>: %{y:.4f}<extra></extra>'
))

# Add confidence intervals
confidence = 1.96 / np.sqrt(len(btc))
fig_acf.add_hline(y=confidence, line_dash="dash", line_color="red", 
                  opacity=0.7, annotation_text="95% CI")
fig_acf.add_hline(y=-confidence, line_dash="dash", line_color="red", opacity=0.7)
fig_acf.add_hline(y=0, line_color="black", opacity=0.5)

fig_acf.update_layout(
    title='Interactive Autocorrelation Function (ACF)',
    xaxis_title='Lag',
    yaxis_title='ACF',
    template='plotly_white',
    height=500
)

fig_acf.show()
print("✓ Interactive ACF plot created!")

### Interactive Multi-Asset Comparison

In [ ]:
# Interactive Multi-Asset Comparison
print("\nCreating multi-asset comparison chart...")

# Normalize prices to start at 100 for comparison
btc_norm = (btc / btc.iloc[0]) * 100
gold_norm = (gold / gold.iloc[0]) * 100
mrf_norm = (mrf / mrf.iloc[0]) * 100

fig_multi = go.Figure()

fig_multi.add_trace(go.Scatter(
    x=btc_norm.index, y=btc_norm.values,
    name='Bitcoin', line=dict(color='orange', width=2),
    hovertemplate='BTC: %{y:.1f}<extra></extra>'
))

fig_multi.add_trace(go.Scatter(
    x=gold_norm.index, y=gold_norm.values,
    name='Gold', line=dict(color='gold', width=2),
    hovertemplate='Gold: %{y:.1f}<extra></extra>'
))

fig_multi.add_trace(go.Scatter(
    x=mrf_norm.index, y=mrf_norm.values,
    name='MRF', line=dict(color='blue', width=2),
    hovertemplate='MRF: %{y:.1f}<extra></extra>'
))

fig_multi.update_layout(
    title='Normalized Price Comparison (Base = 100)',
    xaxis_title='Date',
    yaxis_title='Normalized Price',
    hovermode='x unified',
    template='plotly_white',
    height=500,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig_multi.show()
print("✓ Multi-asset comparison chart created!")

### Summary of Interactive Features

In [ ]:
print("\n" + "="*60)
print("INTERACTIVE VISUALIZATIONS COMPLETE!")
print("="*60)
print("\nBenefits of Interactive Plots:")
print("  ✓ Hover to see exact values and dates")
print("  ✓ Zoom in/out to explore patterns")
print("  ✓ Pan to navigate through time")
print("  ✓ Use range slider for quick navigation")
print("  ✓ Download as static images (camera icon)")
print("  ✓ Compare multiple series easily")

### 📝 Key Takeaway:
- **Interactive plots** enhance data exploration and presentation
- **Plotly** provides easy-to-use interactive visualization tools
- Interactive charts work seamlessly in **Jupyter notebooks and Google Colab**
- Use interactive plots for exploring data; use static plots for reports

---

# ✅ MODULE 5 COMPLETE!

## Summary of Key Concepts:

| Topic | Key Learning |
|-------|-------------|
| 5.1 End-to-End Pipeline | Systematic 8-step approach to time series analysis |
| 5.2 Reusable Functions | Write once, use many times across datasets |
| 5.3 Interactive Viz | Better exploration with Plotly |

---

## 🎯 Your Complete Time Series Toolkit:

After completing all modules, you now have:

1. **Module 3**: Forecasting fundamentals (ARIMA, model selection, diagnostics)
2. **Module 4**: Advanced models (SARIMA, exponential smoothing, VAR, GARCH)
3. **Module 5**: Complete pipeline, reusable code, interactive visualizations

---

## 🚀 Next Steps:

1. Apply the pipeline to your own datasets
2. Experiment with different model parameters
3. Create your own reusable functions
4. Explore more advanced techniques (Machine Learning, Deep Learning)

---

**Congratulations! You now have a complete toolkit for time series analysis!** 🎉